### Ноутбук с расчетом контрфактического анализа. 

Обратите внимание, что расчет трудоемкий и энергоемкий. Ориентировачно занимает два часа, может израсходовать всю батерею на ноутбуке, поэтому советую подключить к зарядке

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import sys

import pickle
filename = 'data/monthly_data.pkl'
with open(filename, 'rb') as f:
    monthly_data = pickle.load(f)
sd_df = pd.read_excel('data/pi_e/s_d.xlsx')

from structuralvar import Clean, SVAR_KL

with open('data/main_hetemodel.pkl', 'rb') as f:
    model, var_spec, data, base_level, raw_data = pickle.load(f)
start_of_row = raw_data.iloc[:12+model.p, 0]/100 + 1

pi_target = [var_spec.transform_to_clean(0.327, var_name='ru_cpi', t_index=t) for t in range(model.T)][model.p:]
def pi_lf(target_series, pi_target):
    return np.mean((target_series[0,:]-pi_target)**2)
cf_result = model.counterfactual_policy(
    policy_shock_index=[2],  # Шоки (в данном случае всего один) политики (ДКП)
    target_series_index=[1],    # Целевая серия
    loss_function=pi_lf,
    n_simulations=1000,# Количество попыток
    ar_order=6,
    pacf_bounds=(-0.8, 0.8),
    verbose=True,
    pi_target = pi_target
)
with open('data/cf_result.pkl', 'wb') as f:
    pickle.dump(cf_result, f)



Минимальное значение функции потерь 0.650817, сделано симуляций 1000/1000 (100.0%)
Оптимальные PACF: [ 0.65391847 -0.56660668 -0.37518525  0.71932132  0.45128444 -0.54124624]
Оптимальные AR коэффициенты: [array([ 1.00136682,  0.50681311, -1.4427317 ,  0.4377485 ,  0.86106782,
       -0.54124624])]
Минимальные потери: 0.650817
